<a href="https://colab.research.google.com/github/Gr1lledChee5e/LLM/blob/main/Day_3_NLP_vs_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load the Data

**Women's E-Commerce Clothing Reviews**

**From Kaggle** - This is a Women's Clothing E-Commerce dataset revolving around the reviews written by customers. Its nine supportive features offer a great environment to parse out the text through its multiple dimensions. Because this is real commercial data, it has been anonymized, and references to the company in the review text and body have been replaced with “retailer”.

https://www.kaggle.com/datasets/nicapotato/womens-ecommerce-clothing-reviews/data


In [ ]:
# Packages
import pandas as pd
import numpy as np

In [ ]:
reviews = pd.read_csv('https://dxl-datasets.s3.us-east-1.amazonaws.com/masllm/reviews.csv')
reviews = reviews[['Review Text', 'Recommended IND']]
reviews['Review Text'] = reviews['Review Text'].values.astype('str') # maybe?
reviews.columns = ['review', 'recommend']
reviews.head()

,review,recommend
0,Absolutely wonderful - silky and sexy and comf...,1
1,Love this dress! it's sooo pretty. i happene...,1
2,I had such high hopes for this dress and reall...,0
3,"I love, love, love this jumpsuit. it's fun, fl...",1
4,This shirt is very flattering to all due to th...,1


# Goals


**I will do two things in this notebook**

1. CLASSIFICATION of reviews into Would / Would Not Recommend
2. SENTIMENT of reviews - Negative, Neutral, Positive

**For each, I will take two different approaches:**

1. Traditional NLP (XGBoost for classification, VADER for sentiment)
2. LLM (OpenAI, Anthropic, Hugging Face)


# Part 1 - NLP Solutions

Without LLMs, NLP tasks like classification and sentiment analysis relied on a mix of feature engineering and machine learning. Here I look at two common traditional NLP problems: classification and sentiment analysis.

NLP methods require us to transform unstructured text into useable features or rely on pre-built sentiment dictionaries; neither of which is required when using LLMs.


In [ ]:
# Packages/ Functions
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report # among many other metrics you could load...

## a) Classification w/ XGBoost

By now you are familiar with basic binary classification models and their use. Here, I will fit an XGBoost model using **recommend** as our target variable.

We need features! Each **review** is currently unstructured text and we need to convert it to something useable. There are *many* different ways to do this, but I will use the popular TF-IDF approach to build a matrix of features from the review text.

I will skip the specifics here - I encourage you to read about it - but the gist of it is this; the TF-IDF procedure results in a sparse matrix where each row represents a review and each column represents a word (one column per word across all reviews), with values that reflect how important that word is to that particular review.

**I am going to use TF-IDF + XGBoost for classification, but consider asking your favorite LLM the following:**

1. Aside from TF-IDF, what are other common approaches to feature engineering with unstructured text? What alternatives exist to TfidVectorizer?
2. Among the many approaches to feature engineering for unstructured text, does one stand out as the most popular or "best"?
3. My professor used XGBoost for a binary classification problem. What other models are commonly used for this task? Can you give me a summary table with the pros and cons, or strengths and weaknesses, of each?


In [ ]:
# x, y train/test split
X_train, X_test, y_train, y_test = train_test_split(reviews['review'], reviews['recommend'], test_size=0.2, random_state=21)

In [ ]:
# X-> currently unstructured text, need to "engineer" useable features

# Term Frequency - Inverse Document Frequency (TF-IDF)
tfidf = TfidfVectorizer(max_features=250, stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [ ]:
# It's hard to view the tfidf results, but here we look at the non-zero elements from the review above
#  The full X matrix is a large, sparse matrix, with a column for each word in the vocabulary (and lots of 0's for words that don't appear in that particular review)
row = X_train_tfidf[0]
nonzero_indices = row.nonzero()[1]
for idx in nonzero_indices:
    print(f"{tfidf.get_feature_names_out()[idx]:20s} → {row[0, idx]:.2f}")





shirt                → 0.22
pictured             → 0.31
color                → 0.18
love                 → 0.14
received             → 0.27
compliments          → 0.28
jeans                → 0.23
fun                  → 0.28
skirt                → 0.22
fits                 → 0.20
true                 → 0.23
size                 → 0.15
normally             → 0.28
wear                 → 0.16
small                → 0.18
retailer             → 0.22
bit                  → 0.20
tight                → 0.24
shoulders            → 0.29


In [ ]:
# XGBoost
model = XGBClassifier(n_estimators=100, max_depth=6, random_state=21)
model.fit(X_train_tfidf, y_train)

# Evaluate
y_pred = model.predict(X_test_tfidf)
print(classification_report(y_test, y_pred, labels=[0, 1], target_names=['Not Recommend', 'Recommend']))

               precision    recall  f1-score   support

Not Recommend       0.57      0.36      0.44        36
    Recommend       0.87      0.94      0.90       164

     accuracy                           0.83       200
    macro avg       0.72      0.65      0.67       200
 weighted avg       0.82      0.83      0.82       200



### **Comments**

1) NLP approach requires unstructured text be converted to useable numeric features, which are then plugged into a model.
2) You lose a lot of context when doing this.
3) TF-IDF can suppress words that are important for this particular task if they appear in many reviews (for example, if all reviews say that "love" the product, love will receive a low weight).
4) How else might you build features for this task other than TF-IDF?

## b) NLP for Sentiment Analysis

For sentiment, there are two common approaches you can take:

1. Download a pre-built sentiment lexicon - pre-built dictionary where the sentiment of each word is scored, then compare with the words in your text.
2. Fit an ML model - Assuming you have labeled data, convert the text to useable features and fit a model.

**Ask an LLM:**

1. I am conducting a sentiment analysis. Is it accurate to say that we can basically go two ways; 1) download a pre-built sentiment lexicon and compare or 2) convert the unstructured text to useable features then fit a model to sentiment, assuming we have labeled data? Am I missing anything? What are the pros and cons of these two approaches?
2. Aside from the VADER lexicon for sentiment analysis, what are the other most common lexicon's and what are they best for?



In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
nltk.download('vader_lexicon')

# Instantiate the "Sentiment Intensity Detector"
sid = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


In [ ]:
sid.polarity_scores('hate')

{'neg': 1.0, 'neu': 0.0, 'pos': 0.0, 'compound': -0.5719}

In [ ]:
# VADER gives score from -1 to 1 (neg to pos)
#   polarity_scores returns score for neg, neu, pos, and compound.
#   these could be used as features in the previous model, completely replacing TF-IDF approach for feature engineering

reviews['vader_compound'] = reviews['review'].apply(lambda x: sid.polarity_scores(x)['compound'])

In [ ]:
# Use that score to classify (into any number of categories desired, here I'll do negative, neutral, positive)
def label_sentiment(score):
    if score >= .7:
        return 'positive'
    elif score <= 0:
        return 'negative'
    else:
        return 'neutral'

reviews['vader_sentiment'] = reviews['vader_compound'].apply(label_sentiment)

In [ ]:
pd.crosstab(reviews['vader_sentiment'], reviews['rec'], margins=True, normalize=True)

rec,0,1,All
vader_sentiment,,,
negative,0.00,0.10,0.1
neutral,0.05,0.15,0.2
positive,0.10,0.60,0.7
All,0.15,0.85,1.0


In [ ]:
#############################################################
# This is NOT part of my intended lecture, I only put this
#  here to see what it would look like to use sentiment scores
#  in place of tf-idf for the classification problem above

# Extract individual VADER sentiment scores into new columns
reviews['vader_neg'] = reviews['review'].apply(lambda x: sid.polarity_scores(x)['neg'])
reviews['vader_neu'] = reviews['review'].apply(lambda x: sid.polarity_scores(x)['neu'])
reviews['vader_pos'] = reviews['review'].apply(lambda x: sid.polarity_scores(x)['pos'])
reviews['vader_compound'] = reviews['review'].apply(lambda x: sid.polarity_scores(x)['compound'])

# XGBoost using vader scores as features
X_train, X_test, y_train, y_test = train_test_split(
    reviews[['vader_neg', 'vader_neu', 'vader_pos', 'vader_compound'] ], reviews['rec'], test_size=0.2, random_state=21
    )

# XGBoost
model = XGBClassifier(n_estimators=100, max_depth=6, random_state=21)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, labels=[0, 1], target_names=['Not Recommend', 'Recommend']))

#############################################################

               precision    recall  f1-score   support

Not Recommend       0.00      0.00      0.00         1
    Recommend       0.75      1.00      0.86         3

     accuracy                           0.75         4
    macro avg       0.38      0.50      0.43         4
 weighted avg       0.56      0.75      0.64         4



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#### **Comments**

1) VADER (and other) pre-built lexicon of words already have sentiment scores, you're basically just doing a lookup for the words in your text then aggregating. Think about limitations here (sarcasm, slang, etc).
2) VADER was built for use in social media text; might not perform as well for product reviews (clothing specific). Others have been built for other purposes, so worth looking around if you find yourself in need.
3) The sentiment scores can be used as-is, or they could be taken as features in a separate model.

# Part 2 - LLM Solutions

* **Traditional NLP** relies heavily on feature engineering; you have unstructured text, can you convert it to something meaningful that can be plugged into a model as features? This leads to a loss of context, as our features are unlikely to capture all the nuances in the language.

* **LLM-based solutions** does not require feature engineering; you pass the text in as-is and the model (hopefully) captures the context and any language-specific nuances.

* Something else to think about - traditional NLP generally requires you train a new model for each new task, whereas LLMs can handle many tasks without explicit training.
  * BUT... if you have the data and can support a good model, it is likely cheaper / faster than making LLM calls.

## a) OpenAI

I will use OpenAI's gpt-5-mini model throughout the course. This should feel familiar if you have used ChatGPT; you ask it something, it gives you a response. Done.

In [ ]:
# Packages
import pandas as pd
from openai import OpenAI
from sklearn.metrics import accuracy_score # or classification_report still fine...

In [ ]:
# Load your API key
from google.colab import userdata
open_ai_key = userdata.get('openai')

In [ ]:
# Define the OpenAI Client
client = OpenAI(api_key = open_ai_key)

In [ ]:
# Experiment with making calls to responses api

#   Simple call
#   Show how to use history
#   Show how to use tools
#   Several cool new features / functionality; skim documentation - https://developers.openai.com/api/docs/guides/migrate-to-responses

# Using Responses API (if you've used their chat completions api before, probably start transitioning to responses...)

response = client.responses.create(
    model='gpt-5-mini',
    instructions='ANALOGOUS TO "instructions" FROM OPENAI',
    input='ANALOGOUS TO INPUT FROM OPENAI',
    previous_response_id = response.id
)



In [ ]:
print(response.output_test)

### gpt-5-mini for Classification

In [ ]:
# Reviews data - I'll grab the first 20 to speed this up.
reviews = pd.read_csv('https://dxl-datasets.s3.us-east-1.amazonaws.com/masllm/reviews.csv')
reviews = reviews[['Review Text', 'Recommended IND']]
reviews['Review Text'] = reviews['Review Text'].values.astype('str')
reviews.columns = ['review', 'rec']
reviews = reviews.iloc[:20, :]
reviews.head()

,review,rec
0,Absolutely wonderful - silky and sexy and comf...,1
1,Love this dress! it's sooo pretty. i happene...,1
2,I had such high hopes for this dress and reall...,0
3,"I love, love, love this jumpsuit. it's fun, fl...",1
4,This shirt is very flattering to all due to th...,1


In [ ]:
# Instructions for the model (want it to return 1 / 0 for Would Recommend / Would Not Recommend)
instructions_classifier = ''

'''

In [ ]:
# Send a single review to check that it's doing what you want it to do.
response = client.responses.create(

)

In [ ]:
print(response.output_text)

In [ ]:
# Create a function that takes a review and returns the response
def get_recommended(text):



In [ ]:
# note that .apply() will send each request sequentially, which can be quite slow.
# alternatives exist to 1) send requests in parallel (asyncio) or 2) in batch (cheaper but slow turnaround, not real-time, this would be good if you had a large job that ran once a day, week, etc..)
reviews['rec_gpt'] = reviews['review'].apply(get_recommended)

In [ ]:
accuracy_score(reviews['rec'], reviews['rec_gpt'].astype('int'))

#### **Quick Aside** - Async to run in parallel

`OpenAI()` sends request then waits before moving on.

`AsyncOpenAI()` sends request but can move on to something else while awaiting response, thus allows for quicker processing if you need to send multiple requests.

*USEFUL FOR THE FIRST HW IF YOU WANT THINGS TO RUN QUICKER*

In [ ]:
import asyncio
from openai import AsyncOpenAI

async_client = AsyncOpenAI(api_key = open_ai_key)

async def get_recommended_async(text):
    response = await async_client.responses.create(
        model='gpt-5-mini',
        instructions=instructions_classifier,
        input=text
    )
    return response.output_text

async def process_all(reviews_list):
    tasks = [get_recommended_async(text) for text in reviews_list]
    return await asyncio.gather(*tasks) # gather = run all these tasks at once, in parallel


In [ ]:
results = await process_all(reviews['review'].tolist())
reviews['rec_gpt_parallel'] = results

In [ ]:
reviews.head()

In [ ]:
reviews = reviews.drop(columns=['rec_gpt_parallel'])

### gpt-5-mini for Sentiment

In [ ]:
# Tailor instructions for sentiment labeling (positive, negative, neutral)
instructions_sentiment = """

"""

def get_sentiment(text):
    response = client.responses.create(
        model = 'gpt-5-mini',
        instructions = instructions_sentiment,
        input = text
    )
    return response.output_text

In [ ]:
reviews['sentiment_gpt'] = reviews['review'].apply(get_sentiment)

In [ ]:
pd.crosstab(reviews['sentiment_gpt'], reviews['rec'], margins=True, normalize=True)

## b) Anthropic



In [ ]:
from anthropic import Anthropic
anthropic_key = userdata.get('claude')

In [ ]:
client_anthropic = Anthropic(api_key = anthropic_key)

In [ ]:
message = client_anthropic.messages.create(
    model='claude-sonnet-4-6',
    max_tokens=10,
    system='ANALOGOUS TO "instructions" FROM OPENAI',
    messages=[
        {"role": "user", "content": 'ANALOGOUS TO INPUT FROM OPENAI'}
    ]
)
print(message.content[0].text)

### Claude for Classification





In [ ]:
# Apply to reviews
#.  Note: Similar to OpenAI, Anthropic has AsyncAnthropic() if you want to run things in parallell
def get_recommended_claude(text):
    message = client_anthropic.messages.create(
        model = 'claude-sonnet-4-6',
        max_tokens = 10,
        system = instructions_classifier,
        messages=[{'role':'user', 'content':text}]
    )
    return message.content[0].text

In [ ]:
reviews['rec_claude'] = reviews['review'].apply(get_recommended_claude)

In [ ]:
# Previously 85% with Open AI
accuracy_score(reviews['rec'], reviews['rec_claude'].astype('int'))

### Claude for Sentiment

In [ ]:
def get_sentiment_claude(text):
    message = client_anthropic.messages.create(
        model = 'claude-sonnet-4-6',
        max_tokens = 10,
        system = instructions_sentiment,
        messages=[{'role':'user', 'content':text}]
    )
    return message.content[0].text

In [ ]:
reviews['sentiment_claude'] = reviews['review'].apply(get_sentiment_claude)

In [ ]:
pd.crosstab(reviews['sentiment_claude'], reviews['sentiment_gpt'])

## c) Hugging Face

Hugging Face is a large open-source community for ML and LLMs. There are thousands of models and datasets available (some good, some not so good, so make sure you are paying attention to the outputs!). They have made it easy to read a model on to your local machine and continue fine-tuning it to your needs, among other things.

For now, we will make use of the `pipeline` class. This takes a `task` or a `model` or both and it handles all the intermediate steps for working with it.

On the site, HuggingFace.co, you can navigate LLMs on the Model tab. You can filter by task and sort them in various ways.

I will look at a few models that seemed relevant or fun:

+ `task = "text-classification"`, default model
+ `task = "zero-shot-classification", model="facebook/bart-large-mnli"`
+ `task = 'text-classification', model='LiYuan/amazon-review-sentiment-analysis'`
+ `task = "text-classification", model="SamLowe/roberta-base-go_emotions"`


In [ ]:
# Load pipeline from transformers package


In [ ]:
# text classification for "positive" vs "negative"
classifier = pipeline(task = 'text-classification', model = 'distilbert/distilbert-base-uncased-finetuned-sst-2-english')

In [ ]:
classifier('')

In [ ]:
# Function to apply (if it returns POSITIVE, convert it to a 1, otherwise a 0)
def get_recommended_hf(text):
    response = classifier(text)
    return 1*(response[0]['label'] == 'POSITIVE')

In [ ]:
reviews['rec_hf'] = reviews['review'].apply(get_recommended_hf)

In [ ]:
accuracy_score(reviews['rec'], reviews['rec_hf'])

In [ ]:
# zero-shot classificaiton for recommend / not recommend
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

In [ ]:
classifier('')

In [ ]:
candidate_labels = ['Yes, I recommend this product', 'No, I do not recommend this product']

def get_recommended_hf2(text):
    response = classifier(
        text,
        candidate_labels
    )
    return 1*(response['labels'][0] == 'Yes, I recommend this product')

In [ ]:
reviews['rec_zeroshot'] = reviews['review'].apply(get_recommended_hf2)

In [ ]:
accuracy_score(reviews['rec'], reviews['rec_zeroshot'])

In [ ]:
reviews[['review', 'rec', 'rec_gpt', 'rec_claude', 'rec_hf', 'rec_zeroshot']]

In [ ]:
# text classification for emotions
emotion_classifier = pipeline(task='text-classification', model = 'SamLowe/roberta-base-go_emotions')
#emotion_classifier = pipeline(task='text-classification', model="bhadresh-savani/distilbert-base-uncased-emotion")

In [ ]:
def get_emotion(text):
    response = emotion_classifier(text)
    return response[0]['label']

In [ ]:
reviews['emotion'] = reviews['review'].apply(get_emotion)

In [ ]:
reviews[['review', 'emotion']].head()

In [ ]:
# Barplot of emotions sorted
import matplotlib.pyplot as plt
emotion_counts = reviews['emotion'].value_counts()
sorted_emotion_counts = emotion_counts.sort_values(ascending=False)

plt.figure(figsize=(10, 6))
plt.bar(sorted_emotion_counts.index, sorted_emotion_counts.values, color='skyblue')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.title('Emotion Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Try another model on your own. Let me know if you find anything particularly fun or useful!**

Think about where traditional ML models might still be preferred and where an LLM approach might. What use cases would each excel at?

**Consider asking an LLM:**

* When comparing traditional ML models to using LLMs for tasks such as binary classification, prediction, sentiment analysis; what are the pros and cons of each? When is one approach preferred over the other? Do traditional ML models still have a place in the world?

## Notebook Summary for GitHub

This notebook explores two main approaches for text analysis on a Women's E-Commerce Clothing Reviews dataset:

1.  **Traditional NLP Solutions**
    *   **Classification with XGBoost:** Demonstrates binary classification (Would Recommend / Would Not Recommend) using TF-IDF for feature engineering and an XGBoost classifier. The `classification_report` is used to evaluate the model.
    *   **Sentiment Analysis with VADER:** Shows how to use the NLTK's VADER lexicon to perform sentiment analysis, generating compound sentiment scores and classifying reviews into positive, neutral, or negative. It also briefly touches upon using VADER scores as features for classification.

2.  **LLM-based Solutions**
    *   **OpenAI:** Utilizes the OpenAI API (specifically `gpt-5-mini`) for both classification and sentiment analysis. It includes examples of synchronous and asynchronous API calls for efficiency.
    *   **Anthropic:** Demonstrates using the Anthropic API (with `claude-sonnet-4-6`) for classification and sentiment analysis, similar to the OpenAI implementation.
    *   **Hugging Face:** Explores various Hugging Face `pipeline` models for text classification, including a default text classifier, a zero-shot classifier, and an emotion classifier. Visualizations are included for emotion distribution.

The notebook highlights the differences between traditional NLP (feature engineering dependent) and LLM-based approaches (direct text input) and their respective applications.